In [0]:
dbutils.widgets.text("server",   "")
dbutils.widgets.text("database", "")
dbutils.widgets.text("username", "")
dbutils.widgets.text("password", "")

In [0]:
from pyspark.sql.functions import lit, current_timestamp
import uuid

server   = dbutils.widgets.get("server")
database = dbutils.widgets.get("database")
username = dbutils.widgets.get("username")
password = dbutils.widgets.get("password")

jdbc_url = f"jdbc:sqlserver://{server}:1433;database={database};encrypt=true;trustServerCertificate=false;"
properties = {
    "user": username,
    "password": password,
    "driver": "com.microsoft.sqlserver.jdbc.SQLServerDriver"
}

batch_id = str(uuid.uuid4())
print(f"Batch ID: {batch_id}")

In [0]:
# Create the bronze volume if it doesn't exist
spark.sql("CREATE SCHEMA IF NOT EXISTS workspace.default")
spark.sql("CREATE VOLUME IF NOT EXISTS workspace.default.bronze")

BRONZE = "/Volumes/workspace/default/bronze"
print(f"Bronze path: {BRONZE}")

In [0]:
tables = ["Employee", "Department", "LeaveRequest", "LeaveBalance", "LeaveType", "AuditLog"]

for table in tables:
    df = spark.read.jdbc(jdbc_url, f"dbo.{table}", properties=properties)
    df = (df
        .withColumn("_src_system",        lit("azure_sql_leave_app"))
        .withColumn("_extracted_at",      current_timestamp())
        .withColumn("_batch_id",          lit(batch_id))
        .withColumn("_is_valid",          lit(True))
        .withColumn("_validation_error",  lit(None).cast("string"))
    )
    df.write.mode("append").parquet(f"{BRONZE}/{table.lower()}")
    print(f"✓ {table}: {df.count()} rows → {BRONZE}/{table.lower()}")

In [0]:
from pyspark.sql.types import StructType, StructField, IntegerType, DateType, StringType, BooleanType
from pyspark.sql import Row
import random
from datetime import date, timedelta

random.seed(42)

# Get active employee IDs from bronze
emp_df = spark.read.parquet(f"{BRONZE}/employee")
active_ids = [r.EmployeeId for r in emp_df.filter("IsActive = true").select("EmployeeId").collect()]

# Generate all working days in 2025 + 2026
def working_days(start, end):
    days = []
    d = start
    while d <= end:
        if d.weekday() < 5:  # Mon–Fri
            days.append(d)
        d += timedelta(days=1)
    return days

all_days = working_days(date(2025, 1, 1), date(2026, 12, 31))

# ~500 rows per employee — sample 500 days if more
rows = []
for emp_id in active_ids:
    sample_days = random.sample(all_days, min(500, len(all_days)))
    for work_date in sample_days:
        is_late    = random.random() < 0.15
        is_early   = random.random() < 0.10
        is_absent  = random.random() < 0.05

        if is_absent:
            rows.append((emp_id, work_date, None, None, False))
            continue

        # Check-in: 08:30–09:00 normal; 09:16–09:45 if late
        if is_late:
            ci_min = 9 * 60 + 16
            ci_max = 9 * 60 + 45
        else:
            ci_min = 8 * 60 + 30
            ci_max = 9 * 60 + 0
        ci = random.randint(ci_min, ci_max)
        check_in = f"{ci // 60:02d}:{ci % 60:02d}"

        # Check-out: 17:00–18:30 normal; 16:00–16:59 if early
        if is_early:
            co_min = 16 * 60 + 0
            co_max = 16 * 60 + 59
        else:
            co_min = 17 * 60 + 0
            co_max = 18 * 60 + 30
        co = random.randint(co_min, co_max)
        check_out = f"{co // 60:02d}:{co % 60:02d}"

        rows.append((emp_id, work_date, check_in, check_out, True))

schema = StructType([
    StructField("EmployeeId",    IntegerType(), False),
    StructField("WorkDate",      DateType(),    False),
    StructField("CheckInTime",   StringType(),  True),
    StructField("CheckOutTime",  StringType(),  True),
    StructField("IsPresent",     BooleanType(), False),
])

att_df = spark.createDataFrame(rows, schema)
att_df = (att_df
    .withColumn("_src_system",       lit("synthetic"))
    .withColumn("_extracted_at",     current_timestamp())
    .withColumn("_batch_id",         lit(batch_id))
    .withColumn("_is_valid",         lit(True))
    .withColumn("_validation_error", lit(None).cast("string"))
)

att_df.write.mode("append").parquet(f"{BRONZE}/attendance")
print(f"✓ Attendance: {att_df.count()} rows → {BRONZE}/attendance")

In [0]:
all_tables = ["employee", "department", "leaverequest", "leavebalance", "leavetype", "auditlog", "attendance"]

for t in all_tables:
    df = spark.read.parquet(f"{BRONZE}/{t}")
    print(f"{BRONZE}/{t}: {df.count()} rows, {len(df.columns)} cols")